In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")


In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [21]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.agents import create_agent          # ← back to this
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model(
    "llama-3.3-70b-versatile",   # ← official replacement for the decommissioned model
    model_provider="groq",
    temperature=0,               # ← MUST be 0 for reliable tool calling
)
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [22]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Based on the search results, here are some recipe suggestions that you can make using leftover chicken and rice:

1. Chicken and Rice Casserole: A classic comfort food dish that combines leftover chicken and rice with creamy sauce, vegetables, and cheese.
2. Chicken and Rice Enchilada Casserole: A Southwestern-inspired dish that uses leftover chicken and rice, along with beans, cheese, and enchilada sauce.
3. Chicken Fried Rice: A quick and easy recipe that uses leftover chicken and rice, along with vegetables, soy sauce, and sesame oil.
4. Leftover Chicken and Egg Fried Rice: A simple and family-friendly meal that combines leftover chicken and rice with eggs, vegetables, and soy sauce.

These are just a few ideas to get you started. You can choose one that sounds appealing to you and follow the recipe instructions to create a delicious meal using your leftover chicken and rice.

Would you like me to provide you with the recipe instructions for any of these dishes?


In [23]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='f5ca4d23-93b9-4dd9-92e8-e9fde4f26059'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gfcsrgwxw', 'function': {'arguments': '{"query":"recipes using leftover chicken and rice"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 282, 'total_tokens': 302, 'completion_time': 0.048520091, 'completion_tokens_details': None, 'prompt_time': 0.026870198, 'prompt_tokens_details': None, 'queue_time': 0.153195724, 'total_time': 0.075390289}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f39a8-b9c0-72d0-b1cb-572703ad6b48-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'recipes using leftover ch